# 实验 1：解剖 Qwen3-0.6B-Base 的静态结构

## 目标

像拆开一台机器一样，观察这个模型由哪些部件组成，以及每个部件的尺寸。

本实验会加载约 1.2GB 的原始权重，但**不生成文本、不训练模型**。

```text
token IDs
  ↓ embed_tokens
1024 维内部向量
  ↓ 28 个 Transformer block
结合上下文后的 1024 维向量
  ↓ lm_head
151,936 个候选下一个 token 的分数
```


## 外部插图：如何使用这些提示词

本 Notebook 不生成或下载图片。下面各节的“插图提示词”可以直接交给你使用的图片生成 AI；生成后再把图片手动插入相应位置。

### 统一视觉规范

所有图片都采用相同的技术教材风格：**16:9，2048 × 1152，白色或暖灰背景，扁平二维矢量信息图，网格对齐，大面积留白**。用深色无衬线字体；标题至少 44 px，模块标签至少 28 px，关键数字至少 32 px。颜色语义固定：**海军蓝 = 主数据流，青绿色 = attention，橙色 = MLP，紫色 = 权重共享，灰色 = RMSNorm/辅助结构**。

不要生成人物、机器人、装饰性芯片、电路板、3D 渲染、渐变、公式云、水印或与本模型无关的组件。生成式图片模型经常画错文字和数字；**本 Notebook 中的提示词、代码和实测输出才是结构事实的权威来源**。生成图片后，逐项核对尺寸、头数和箭头方向；必要时用矢量编辑器修正文字。

In [1]:
from pathlib import Path

import torch
import transformers
from transformers import AutoModelForCausalLM


def find_project_root() -> Path:
    for directory in (Path.cwd(), *Path.cwd().parents):
        if (directory / 'models' / 'Qwen3-0.6B-Base').is_dir():
            return directory
    raise FileNotFoundError('找不到 models/Qwen3-0.6B-Base')


PROJECT_ROOT = find_project_root()
MODEL_PATH = PROJECT_ROOT / 'models' / 'Qwen3-0.6B-Base'

print(f'项目根目录: {PROJECT_ROOT}')
print(f'模型目录: {MODEL_PATH}')
print(f'PyTorch: {torch.__version__}')
print(f'Transformers: {transformers.__version__}')


项目根目录: /home/linjunjie/Workspace/xxdw1
模型目录: /home/linjunjie/Workspace/xxdw1/models/Qwen3-0.6B-Base
PyTorch: 2.13.0+cu130
Transformers: 5.15.1


In [2]:
# 本实验只观察结构，因此固定在 CPU 上加载。
# 这样不受当前 GPU Triton 编译器环境的影响。
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    dtype=torch.bfloat16,
).cpu().eval()

print(f'模型类: {type(model).__name__}')
print(f'模型设备: {next(model.parameters()).device}')
print(f'权重数据类型: {next(model.parameters()).dtype}')


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

模型类: Qwen3ForCausalLM
模型设备: cpu
权重数据类型: torch.bfloat16


## 顶层骨架

`Qwen3ForCausalLM` 是完整的“预测下一个 token”模型。它的主体 `model` 负责把 token 的内部向量加工 28 次；`lm_head` 则把最终的 1024 维向量映射到整个词表。

注意：以下所有 block 的主通道输入和输出都是 1024 维。这是模型内部信息流动的固定宽度。


In [3]:
config = model.config

print(f'总参数量: {sum(parameter.numel() for parameter in model.parameters()):,}')
print(f'词表大小: {config.vocab_size:,}')
print(f'隐藏向量维度: {config.hidden_size}')
print(f'Transformer block 数量: {config.num_hidden_layers}')
print(f'注意力头: Q={config.num_attention_heads}, KV={config.num_key_value_heads}')
print(f'每头维度: {config.head_dim}')
print(f'MLP 中间维度: {config.intermediate_size}')

print('\nQwen3ForCausalLM')
print('├── model: Qwen3Model')
print(f'│   ├── embed_tokens: Embedding({config.vocab_size}, {config.hidden_size})')
print(f'│   ├── layers: ModuleList × {len(model.model.layers)}')
print(f'│   └── norm: {model.model.norm}')
print(f'└── lm_head: Linear({config.hidden_size}, {config.vocab_size}, bias=False)')


总参数量: 596,049,920
词表大小: 151,936
隐藏向量维度: 1024
Transformer block 数量: 28
注意力头: Q=16, KV=8
每头维度: 128
MLP 中间维度: 3072

Qwen3ForCausalLM
├── model: Qwen3Model
│   ├── embed_tokens: Embedding(151936, 1024)
│   ├── layers: ModuleList × 28
│   └── norm: Qwen3RMSNorm((1024,), eps=1e-06)
└── lm_head: Linear(1024, 151936, bias=False)


### 插图提示词 1：顶层信息流

**学习目标：** 把 Python 模块树对应到从 token ID 到输出 logits 的实际计算路径。

```text
请创建一张用于机器学习教材的 Qwen3-0.6B-Base 顶层架构信息图，横向 16:9，2048×1152，白色或暖灰背景，扁平二维矢量风格，深色无衬线字体，网格对齐，大面积留白。使用海军蓝表示主数据流，青绿色表示 attention，橙色表示 MLP，紫色表示共享权重，灰色表示 RMSNorm；不要人物、机器人、芯片、电路装饰、3D、渐变、水印或无关组件。

从左到右准确画出四个主要阶段，并使用清晰箭头连接：
1. 左侧是一小串离散的 token ID 方块，标注“token IDs”；
2. 进入 token embedding 查表，标注“Embedding(151,936, 1024)”以及“151,936 个词表条目 × 1024 维”，输出标注“1024 维 hidden vector”；
3. 进入一个纵向串行的重复模块堆，标注“28 sequential Transformer blocks”，用少量重复卡片和省略号表示，但明确写出“28”；每个 block 的输入和输出都必须标注为“1024 维”，不能画成逐层变宽；
4. 经过“final RMSNorm(1024)”后进入“lm_head: Linear(1024, 151,936, bias=False)”，输出标注“151,936 next-token logits”。不要把 logits 标成概率，不要添加 softmax。

在图的侧边增加一张简洁参数卡，只写：
“Parameters: 596,049,920”
“Vocabulary: 151,936”
“Hidden size: 1024”
“Layers: 28”

用一条紫色的共享权重连线从输入 embedding 指向 lm_head，并标注“same shared vocabulary weight matrix”。这条线只表示权重共享，不表示激活绕过 Transformer。所有数字和模块名称必须逐字校对；中文标签请保留足够空间，避免使用过小文字。
```

**生成后核对：**

- 是否确实是 28 个串行 block，且每层前后都是 1024 维？
- 是否把 `lm_head` 输出画成 logits，而不是概率？
- 是否明确表示 embedding 与 `lm_head` 共享权重，而非两份矩阵？

## 拆开一个 Transformer block

28 个 block 的结构相同，所以先观察第 0 层就够了。

一个 block 的主路径是：

```text
输入 → RMSNorm → Self-Attention → 残差相加
     → RMSNorm → MLP            → 残差相加 → 输出
```

这里的 Self-Attention 让不同 token 交换信息；MLP 则单独加工每个 token 的内部特征。


In [7]:
layer = model.model.layers[0]
print(layer)


Qwen3DecoderLayer(
  (self_attn): Qwen3Attention(
    (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
    (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
    (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
    (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
    (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
    (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
  )
  (mlp): Qwen3MLP(
    (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
    (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
    (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
    (act_fn): SiLUActivation()
  )
  (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
  (post_attention_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
)


### 插图提示词 2：一个 Qwen3 Transformer block 的残差路径

**学习目标：** 看清一个 block 内 attention 与 MLP 的先后顺序，以及两次残差连接如何维持 1024 维主通道。

```text
请创建一张单层 Qwen3-0.6B-Base decoder block 的精确教学信息图，横向 16:9，2048×1152，白色或暖灰背景，扁平二维矢量技术教材风格。海军蓝表示 1024 维 residual stream，青绿色表示 self-attention，橙色表示 MLP，灰色表示 RMSNorm；使用深色无衬线字体、规则网格和清晰左到右箭头。不要人物、机器人、芯片、电路装饰、3D、渐变、水印或未列出的模块。

严格画出 pre-norm 残差顺序：
“输入 x (1024 维)” → “RMSNorm(1024)” → “Self-Attention” → 第一个加号，和原始输入 residual 相加 → “RMSNorm(1024)” → “Gated MLP” → 第二个加号，和更新后的 residual 相加 → “输出 (1024 维)”。

必须有两条清晰的弯曲 skip connection，分别绕过 attention 分支和 MLP 分支，并在两处都使用“+”号。主数据流在输入、两个加号之间、输出处都保持“1024 维”。不要将 attention 和 MLP 画成并行后只相加一次。

在 Self-Attention 模块内部用小型子图说明：多个 token 位置彼此连线，表达跨 token 信息交换；从 1024 维输入生成：
- Q: 1024 → 2048 = 16 query heads × 128，之后 q_norm(128)；
- K: 1024 → 1024 = 8 key heads × 128，之后 k_norm(128)；
- V: 1024 → 1024 = 8 value heads × 128；
- GQA attention 后，16 × 128 = 2048，再经 o_proj: 2048 → 1024。

在 Gated MLP 模块内部准确画出两条并行分支：
- gate_proj: 1024 → 3072，然后 SiLU；
- up_proj: 1024 → 3072；
两支经过逐元素乘法“⊙”，维度仍为 3072，最后 down_proj: 3072 → 1024。MLP 内不得画任何 token 与 token 之间的连线，强调每个 token 被相同变换独立处理。所有尺寸、箭头和模块文字必须逐字校对；为中文注释留出空间。
```

**生成后核对：**

- 是否有两次 RMSNorm、两次残差相加，且顺序正确？
- attention 是否表现 token 间通信，而 MLP 没有 token 间连线？
- Q/K/V 总宽度是否分别为 2048/1024/1024，MLP 是否为 1024→3072→1024？

## 关键权重矩阵

PyTorch 的线性层权重形状写作 `(输出维度, 输入维度)`。例如 Q 投影的 `(2048, 1024)` 表示：每个 1024 维输入向量会被变换为 2048 维 Q 向量。

Qwen3-0.6B 使用 GQA（Grouped-Query Attention）：16 个 Query 头，但只有 8 个 Key 头和 8 个 Value 头。因为每头 128 维，所以 Q 是 `16 × 128 = 2048` 维，而 K/V 各是 `8 × 128 = 1024` 维。


In [9]:
parameters = (
    ('token embedding', model.model.embed_tokens.weight),
    ('Q projection', layer.self_attn.q_proj.weight),
    ('K projection', layer.self_attn.k_proj.weight),
    ('V projection', layer.self_attn.v_proj.weight),
    ('attention output projection', layer.self_attn.o_proj.weight),
    ('MLP gate projection', layer.mlp.gate_proj.weight),
    ('MLP up projection', layer.mlp.up_proj.weight),
    ('MLP down projection', layer.mlp.down_proj.weight),
    ('language-model head', model.lm_head.weight),
)

for name, parameter in parameters:
    print(f'{name:28} {tuple(parameter.shape)}')

shared_weights = (
    model.model.embed_tokens.weight.data_ptr()
    == model.lm_head.weight.data_ptr()
)
print(f'\n输入 embedding 与 lm_head 是否共享同一块权重: {shared_weights}')


token embedding              (151936, 1024)
Q projection                 (2048, 1024)
K projection                 (1024, 1024)
V projection                 (1024, 1024)
attention output projection  (1024, 2048)
MLP gate projection          (3072, 1024)
MLP up projection            (3072, 1024)
MLP down projection          (1024, 3072)
language-model head          (151936, 1024)

输入 embedding 与 lm_head 是否共享同一块权重: True


### 插图提示词 3：GQA 的 16 个 Query 头与 8 组 K/V 头

**学习目标：** 不计算注意力分数，只观察头的数量和共享关系，理解为什么 Q、K、V 的投影宽度不同。

```text
请创建一张用于机器学习教材的 GQA（Grouped-Query Attention）教学信息图，横向 16:9，2048×1152，白色或暖灰背景，扁平二维矢量风格，深色无衬线字体，规则网格，大面积留白。使用海军蓝表示 Query，青绿色表示 Key/Value，灰色表示共享连线；不要人物、机器人、芯片、电路装饰、3D、渐变、水印、注意力热力图、softmax 或未说明的计算步骤。

主标题写：“GQA：16 个 Query 头共享 8 组 Key/Value 头”。画面左右两栏，中间是明确的配对连线。

左栏标题：“Query：16 个头，每头 128 维”。垂直排列 16 个小圆角框：
“Q0（128）”“Q1（128）”“Q2（128）”“Q3（128）”
“Q4（128）”“Q5（128）”“Q6（128）”“Q7（128）”
“Q8（128）”“Q9（128）”“Q10（128）”“Q11（128）”
“Q12（128）”“Q13（128）”“Q14（128）”“Q15（128）”。
左栏底部写：“Q 总宽度：16 × 128 = 2048”。

右栏标题：“Key / Value：8 组，每个头 128 维”。垂直排列 8 个成对框：
“KV0：K0（128） + V0（128）”
“KV1：K1（128） + V1（128）”
一直到“KV7：K7（128） + V7（128）”。
右栏底部写：“K 总宽度：8 × 128 = 1024”和“V 总宽度：8 × 128 = 1024”。

中间用编号文字和细虚线明确表示共享关系：
“Q0、Q1 → KV0”，“Q2、Q3 → KV1”，“Q4、Q5 → KV2”，“Q6、Q7 → KV3”，
“Q8、Q9 → KV4”，“Q10、Q11 → KV5”，“Q12、Q13 → KV6”，“Q14、Q15 → KV7”。
在底部醒目说明框写：“每 2 个 Query 头共享 1 组 Key/Value 头”“这就是 Grouped-Query Attention（GQA）”。

严格技术要求：必须是 16 个 Q 头、8 组 KV；每一组 KV 恰好被两个相邻的 Q 头共享；K 与 V 必须保持为分别的 8 个 128 维头，不能合并成一个 256 维头，不能错误画为 16 个独立 K/V 头。所有中文、编号和数字必须清晰且逐字正确。
```

**生成后核对：**

- Q0/Q1 到 Q14/Q15 是否恰好两两映射到 8 组 KV？
- 是否分别保留 `K 总宽度 = 1024` 和 `V 总宽度 = 1024`？
- 是否没有把 K/V 错画为 16 个独立头，或合并为 256 维头？

### 可选插图提示词 4：输入 embedding 与 lm_head 的权重共享

**学习目标：** 在完成前三张图后，理解为什么实验检查到 `embed_tokens.weight` 与 `lm_head.weight` 指向同一块存储。

```text
请创建一张简洁的权重共享教学信息图，横向 16:9，2048×1152，白色或暖灰背景，扁平二维矢量技术教材风格，深色无衬线字体，规则网格和留白。使用紫色表示共享权重，海军蓝表示主箭头；不要人物、机器人、芯片、电路装饰、3D、渐变、水印、softmax、训练损失或任何未列出的组件。

主标题写：“权重共享：同一块词表矩阵用于输入和输出”。把一块紫色的大矩阵放在画面中央，矩阵标签必须是：
“共享词表矩阵 E”
“存储形状：(151,936, 1024)”
“同一块权重，不是两份复制矩阵”。

从左侧画输入路径：
“token ID” → “查表：取 E 的一行” → “1024 维 embedding”。

从右侧画输出路径：
“最终 hidden vector h（1024 维）” → “h × Eᵀ” → “151,936 个 logits”。

在矩阵旁增加一个小验证标签：
“embed_tokens.weight 与 lm_head.weight”
“data_ptr 相同：True”。

严格技术要求：中央只能出现一块矩阵 E；输入侧必须表示从 E 中查一行；输出侧必须明确写 `h × Eᵀ`，不能写成 `E × h`；输出只能写 logits，不能写概率；不要画第二份 lm_head 权重矩阵。所有文字和数字必须逐字清晰、准确；若无法可靠生成中文文字，请保留清晰标签框供人工替换。
```

**生成后核对：**

- 是否只有一块 `(151,936, 1024)` 的词表矩阵？
- 输出计算是否明确为 `h × Eᵀ`？
- 是否保留“logits，不是概率”的正确表述？

## 小结与思考

你已经从真实权重中确认：Qwen3-0.6B 有 28 个重复 block，内部主通道宽度是 1024，attention 使用 16 个 Q 头与 8 个 KV 头，MLP 会把 1024 维暂时扩展到 3072 维再压回 1024 维。

试着回答这三个问题：

1. 为什么 28 个 block 都保持输入和输出为 1024 维，而不是每层都继续变宽？
2. 为什么 Q 是 2048 维，但 K 和 V 各只有 1024 维？
3. 为什么模型可以让输入 embedding 和输出 lm_head 共享同一块词表矩阵？

下一实验会亲手观察一个 token ID 如何从 `embed_tokens` 变成 1024 维向量。
